<a href="https://colab.research.google.com/github/fidlarsyn/Scikit-learn-Cookbook--O-Reilly-/blob/main/2_Pre_Model_Workflow_and_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **The impact of raw data on model performance**
Algoritme machine learning dirancang untuk mempelajari pola dari data. Jika data input cacat—baik karena data hilang, adanya outlier, atau fitur yang tidak relevan—kemampuan model untuk melakukan generalisasi pada data baru akan berkurang drastis.

* **Masalah Umum**: Data yang "kotor" dapat menyebabkan model membuat asumsi yang salah, menghasilkan prediksi yang tidak akurat, dan memperburuk pengambilan keputusan.
* **Prinsip Utama**: Berlaku istilah "garbage in, garbage out" (sampah yang masuk akan menghasilkan sampah pula), sehingga prapemrosesan bukanlah tugas sekali jalan, melainkan proses berkelanjutan.

# **Handling missing data**
Sebagian besar algoritme scikit-learn tidak dapat dijalankan jika terdapat nilai yang hilang (NaN) dalam data. scikit-learn menyediakan tiga strategi imputasi utama:
* **SimpleImputer**: Mengganti nilai hilang dengan statistik sederhana seperti rata-rata (mean), median, atau nilai yang paling sering muncul.
* **KNNImputer**: Menggunakan algoritme K-Nearest Neighbors untuk mengisi nilai berdasarkan kemiripan dengan sampel tetangga terdekat.
* **IterativeImputer**: Metode tingkat lanjut yang memodelkan setiap fitur dengan nilai hilang sebagai fungsi regresi dari fitur lainnya secara berulang.

**Contoh Kode Imputasi:**

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import pandas as pd

# 1. Simple Imputer (Rata-rata)
imputer = SimpleImputer(strategy="mean")
imputed_data = imputer.fit_transform(df)

# 2. KNN Imputer (Berdasarkan 2 tetangga terdekat)
knn_imputer = KNNImputer(n_neighbors=2)
knn_data = knn_imputer.fit_transform(df)

# 3. Iterative Imputer (Multivariat)
iter_imputer = IterativeImputer()
iter_data = iter_imputer.fit_transform(df)

# **Scaling Techniques**
Fitur sering kali memiliki rentang nilai yang sangat berbeda (misal: usia 0–100 vs. pendapatan 0–1.000.000). Penskalaan memastikan tidak ada satu fitur pun yang mendominasi proses pembelajaran.
* **StandardScaler**: Mengubah data agar memiliki rata-rata 0 dan standar deviasi 1 (z-score). Cocok untuk data berdistribusi normal.
* **MinMaxScaler**: Mengubah rentang data menjadi skala tetap, biasanya antara 0 dan 1.
* **Normalizer**: Menskalakan setiap sampel individu agar memiliki norma unit (L1 atau L2), berguna untuk data yang jarang (sparse).

**Contoh Kode Scaling:**

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer

# Standardisasi
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# Min-Max Scaling (0 ke 1)
minmax = MinMaxScaler()
X_minmax = minmax.fit_transform(df)

# Normalisasi (Norma L2)
normalizer = Normalizer()
X_normalized = normalizer.fit_transform(df)

# **Encoding categorical variables**
Komputer dan algoritme ML memerlukan input numerik, sehingga variabel teks/kategori harus dikonversi.
* **OneHotEncoder**: Membuat kolom biner (0 dan 1) untuk setiap kategori. Cocok untuk data nominal tanpa urutan (misal: warna).
* **LabelEncoder**: Memberikan integer unik ke setiap kategori. Digunakan dengan hati-hati karena dapat menyiratkan urutan yang sebenarnya tidak ada.
* **ColumnTransformer**: Memungkinkan Anda menerapkan transformasi yang berbeda pada kolom yang berbeda secara bersamaan.

**Contoh Kode Encoding & Transformer:**

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Menggabungkan transformasi numerik dan kategorikal
column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary"]),        # Skalakan angka
        ("cat", OneHotEncoder(), ["Dept", "Position"])      # Encode kategori
    ],
    remainder="passthrough" # Lewati kolom lainnya
)
transformed_data = column_transformer.fit_transform(mixed_data)

# **Introduction to pipelines in scikit-learn**
Pipeline adalah urutan langkah pemrosesan yang dieksekusi secara berurutan.
* **Keuntungan**: Menyederhanakan kode, menjaga konsistensi transformasi antara data latih dan uji, serta mencegah kebocoran data (data leakage).
* **Alur Kerja**: Anda dapat menggabungkan imputasi, penskalaan, dan model akhir ke dalam satu objek tungga.

**Contoh Kode Pipeline:**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Membangun rantai kerja otomatis
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")), # Langkah 1
    ("scaler", StandardScaler()),                # Langkah 2
    ("model", LogisticRegression())              # Langkah 3 (Estimator)
])

# Melatih seluruh pipeline sekaligus
pipeline.fit(X_train, y_train)

# **Feature Engineering**
Ini mencakup pembuatan fitur baru (ekstraksi) dan pemilihan fitur yang relevan (seleksi) untuk meningkatkan performa model.
* **PolynomialFeatures**: Membuat fitur baru dari kombinasi polinomial dan interaksi antar fitur numerik.
* **KBinsDiscretizer**: Mengubah variabel kontinu menjadi kategori dengan membaginya ke dalam beberapa "ember" (bins).
* **RFE** (Recursive Feature Elimination): Menghapus fitur yang paling tidak penting secara bertahap berdasarkan peringkat kepentingan dari model.

**Contoh Kode Feature Engineering:**

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, KBinsDiscretizer
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

# Membuat fitur polinomial derajat 2
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

# Binning data kontinu menjadi 3 bagian
kbins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform")
X_binned = kbins.fit_transform(X)

# Seleksi fitur otomatis (memilih 1 fitur terbaik)
rfe = RFE(estimator=LinearRegression(), n_features_to_select=1)
rfe.fit(X, y)